# Menstrual-cycle phase classification

Decision-tree and random-forest coursework using hormones and wrist temperature from mcPHASES version 1.0.0. The target is cycle phase. The original split is by measurement row, so results do not demonstrate generalisation to new participants. See [README](README.md) for required data, access conditions and methodological limits.


## Imports and data

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Load tables
hormones_df = pd.read_csv("hormones_and_selfreport.csv")
wrist_df    = pd.read_csv("wrist_temperature.csv")


# Accept the two temperature-column spellings used in the source notebooks.
if 'temperature_diff_from_baseline' not in wrist_df.columns and 'temp_diff_from_baseline' in wrist_df.columns:
    wrist_df = wrist_df.rename(columns={'temp_diff_from_baseline':'temperature_diff_from_baseline'})

# Retain study interval in the join when both source tables provide it.
merge_keys = ["id", "day_in_study"]
if "study_interval" in hormones_df.columns and "study_interval" in wrist_df.columns:
    merge_keys.insert(1, "study_interval")

# Keep the required columns.
hormones_df = hormones_df[merge_keys + ["phase", "lh", "estrogen", "pdg"]]
wrist_df    = wrist_df[merge_keys + ["temperature_diff_from_baseline"]]

# Merge on id + day_in_study
df_wrist = hormones_df.merge(
    wrist_df,
    on=merge_keys,
    how="inner"
)

# Drop rows with missing key values
df_wrist = df_wrist.dropna(subset=[
    "phase",
    "lh", "estrogen", "pdg",
    "temperature_diff_from_baseline"
])

print("Merged shape:", df_wrist.shape)
print(df_wrist.head())

# Define a consistent phase order if you want
phase_order = ["Menstrual", "Follicular", "Fertility", "Luteal"]


## Statistical analysis

In [ ]:
def kruskal_by_phase(data, outcome, phase_col="phase", order=None):
    if order is None:
        order = sorted(data[phase_col].dropna().unique())
    groups = [data.loc[data[phase_col] == p, outcome].dropna() for p in order]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        stat, p = stats.kruskal(*groups)
        return stat, p
    return np.nan, np.nan

stat, p_kw = kruskal_by_phase(
    df_wrist,
    outcome="temperature_diff_from_baseline",
    order=phase_order
)

print("Kruskal–Wallis wrist temp across phases:")
print("H =", stat, "p =", p_kw)

In [ ]:
summary_rows = []
for phase in phase_order:
    vals = df_wrist.loc[df_wrist["phase"] == phase,
                        "temperature_diff_from_baseline"].dropna()
    if len(vals) == 0:
        continue
    n = len(vals)
    mean = vals.mean()
    sd = vals.std(ddof=1)
    sem = sd / np.sqrt(n)
    ci_low, ci_high = stats.t.interval(
        0.95, df=n-1, loc=mean, scale=sem
    )
    summary_rows.append({
        "phase": phase,
        "n": n,
        "mean_temp": mean,
        "ci_low": ci_low,
        "ci_high": ci_high
    })

wrist_ci = pd.DataFrame(summary_rows)
print("\nWrist temperature by phase (mean ± 95% CI):")
print(wrist_ci)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

phase_order = ["Menstrual", "Follicular", "Fertility", "Luteal"]
wrist_ci["phase"] = pd.Categorical(wrist_ci["phase"],
                                   categories=phase_order,
                                   ordered=True)
wrist_ci = wrist_ci.sort_values("phase")

x = np.arange(len(wrist_ci))
means = wrist_ci["mean_temp"].values
# true half‑widths
low = means - wrist_ci["ci_low"].values
high = wrist_ci["ci_high"].values - means
# Display the actual calculated intervals without enlarging them.
factor = 1
low_plot = low * factor
high_plot = high * factor

plt.figure(figsize=(6,4))
plt.scatter(x, means, color="#d33682", zorder=3)

plt.errorbar(
    x,
    means,
    yerr=[low_plot, high_plot],
    fmt="none",
    ecolor="#d33682",
    elinewidth=1.5,
    capsize=4,
    zorder=2,
)

plt.axhline(0, color="gray", linestyle="--", linewidth=0.5)
plt.xticks(x, wrist_ci["phase"].astype(str))
# Allow the axis to include the full confidence intervals.
plt.ylabel("Temperature difference from baseline (°C)")
plt.xlabel("Menstrual phase")
plt.title("Nightly wrist temperature difference by phase\n(mean ± 95% CI)")
plt.tight_layout()
plt.show()

In [ ]:
corr_results = []
for hormone in ["lh", "estrogen", "pdg"]:
    tmp = df_wrist[[hormone, "temperature_diff_from_baseline"]].dropna()
    if len(tmp) > 2:
        rho, p = stats.spearmanr(tmp[hormone],
                                 tmp["temperature_diff_from_baseline"])
        corr_results.append({
            "hormone": hormone,
            "n": len(tmp),
            "spearman_rho": rho,
            "p_value": p
        })

corr_results_df = pd.DataFrame(corr_results)
print("\nSpearman correlations: hormones vs wrist temp")
print(corr_results_df)

## Decision tree

In [ ]:
feature_cols = [
    "lh",
    "estrogen",
    "pdg",
    "temperature_diff_from_baseline"
]

X = df_wrist[feature_cols]
y = df_wrist["phase"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,      # keep phase proportions
    random_state=42
)
 
tree_clf = DecisionTreeClassifier(
    criterion="gini",
    class_weight="balanced",
    max_depth=3,     # shallow tree, easier to interpret
    random_state=42
)

tree_clf.fit(X_train, y_train)
y_pred = tree_clf.predict(X_test)

print("\n=== Decision tree: hormones + wrist temp ===")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

rules = export_text(tree_clf, feature_names=feature_cols)
print("\nTree rules:\n")
print(rules)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred, labels=phase_order)
from matplotlib.colors import LinearSegmentedColormap

pink_cmap = LinearSegmentedColormap.from_list(
   "soft_pink",
    ["#fff5f7", "#f48fb1", "#ad1457"]  # very light -> medium pink -> dark
)

plt.figure(figsize=(6, 5))
sns.heatmap(cm,
 annot=True,
   fmt="d",
    cmap=pink_cmap,
    xticklabels=phase_order,
    yticklabels=phase_order
)
plt.xlabel("Predicted phase")
plt.ylabel("True phase")
plt.title("Decision tree confusion matrix (hormones + wrist temperature)")
plt.tight_layout()
plt.show()

## Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 1. Features and target
feature_cols = ["lh", "estrogen", "pdg", "temperature_diff_from_baseline"]
X = df_wrist[feature_cols]
y = df_wrist["phase"]

# 2. Train–test split (keep phase proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 3. Random forest model
rf_clf = RandomForestClassifier(
    n_estimators=200,   # number of trees
    max_depth=None,     # allow deeper trees
    random_state=42,
    n_jobs=-1,          # use all CPU cores
)

# 4. Fit and predict
rf_clf.fit(X_train, y_train)
y_pred = rf_clf.predict(X_test)

# 5. Evaluation
print("\n=== Random forest: hormones + wrist temp ===")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1) Custom soft‑pink colormap
pink_cmap = LinearSegmentedColormap.from_list(
    "soft_pink",
    ["#fff5f7", "#f48fb1", "#ad1457"]  # very light -> medium -> dark pink
)

# 2) Compute confusion matrix (assuming y_test, y_pred already exist)
cm = confusion_matrix(y_test, y_pred, labels=phase_order)

# 3) Plot heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap=pink_cmap,
    xticklabels=phase_order,
    yticklabels=phase_order
)
plt.xlabel("Predicted phase")
plt.ylabel("True phase")
plt.title("Random forest confusion matrix (hormones + wrist temperature)")
plt.tight_layout()
plt.show()

## Temperature-only random forest

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# 1. Select features and target
X = df_wrist[['temperature_diff_from_baseline']].copy()   # only temperature
y = df_wrist['phase'].copy()          # 4-class phase label

# 2. Encode phase labels to integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Train–test split (stratified by phase)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# 4. Random forest with only temp as predictor
rf_temp_only = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_temp_only.fit(X_train, y_train)

# 5. Evaluate
y_pred = rf_temp_only.predict(X_test)

print("Classification report (temp-only → phase):")
print(classification_report(y_test, y_pred, labels=np.arange(len(le.classes_)), target_names=le.classes_, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
